In [5]:
import os
import re
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer

In [6]:
tasks = [
    ("Assin2STS", None, "test"),  # sentence1, sentence2
    ("SICK-BR-STS", None, "test"),  # sentence1, sentence2
    ("stsb_multi_mt", "pt", "test"),  # sentence1, sentence2

    ("MassiveIntentClassification", "pt", "test"),  # text
    ("multi_hate", "por", "test"),  # text
    ("BrazilianToxicTweetsClassification", None, "test"),  # text
    ("HateSpeechPortugueseClassification", None, "train"),  # text
    ("tweet_sentiment_multilingual", "portuguese", "test"),  # text

    ("MultiLongDocReranking", "pt-corpus", "test"),  # text
    ("WikipediaRerankingMultilingual", "pt-corpus", "test"),  # text
    ("XGlueWPRReranking", "pt-corpus", "test"),  # text

    ("WebFAQRetrieval", "por-corpus", "test"),  # text
    ("MultiLongDocRetrieval", "pt-corpus", "test"),  # text
    ("WikipediaRetrievalMultilingual", "pt-corpus", "test"),  # text
]

# Columns to consider (based on your comments)
COLUMNS_BY_DATASET = {
    "Assin2STS": ["sentence1", "sentence2"],
    "SICK-BR-STS": ["sentence1", "sentence2"],
    "stsb_multi_mt": ["sentence1", "sentence2"],
    # everything else: "text"
}

DEFAULT_COLUMNS = ["text"]

MODEL_NAME = "iara-project/BERTimbau-large-matryoshka-sts-pt"  # <-- change to your model
OUTPUT_DIR = "../data/tokens/bertimbau"
STREAMING = True               # safer for very large datasets
MAX_STORED_LENGTHS = 200_000   # cap stored lengths for percentile estimates (reservoir sampling)

In [7]:
def safe_name(s: str) -> str:
    s = str(s)
    s = s.strip().replace("/", "_")
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", s)
    return s


def to_text(val):
    """Convert dataset cell to a string suitable for tokenization."""
    if val is None:
        return None
    if isinstance(val, str):
        return val
    if isinstance(val, (list, tuple)):
        # join lists of strings (common in some datasets)
        parts = []
        for x in val:
            if x is None:
                continue
            parts.append(str(x))
        return " ".join(parts) if parts else None
    return str(val)


def token_len(tokenizer, text: str) -> int:
    # truncation=False so you measure true length, even if model later truncates at inference
    enc = tokenizer(text, add_special_tokens=True, truncation=False)
    return len(enc["input_ids"])


def reservoir_update(reservoir, new_value, seen_count, cap):
    """Reservoir sampling to keep a uniform sample up to 'cap' items."""
    if len(reservoir) < cap:
        reservoir.append(new_value)
    else:
        j = np.random.randint(0, seen_count)
        if j < cap:
            reservoir[j] = new_value


def compute_stats_for_field(ds_iter, field, tokenizer, max_store=MAX_STORED_LENGTHS):
    n = 0
    s = 0.0
    s2 = 0.0
    mn = math.inf
    mx = -math.inf
    stored = []

    for row in ds_iter:
        if field not in row:
            raise KeyError(f"Field '{field}' not found. Available columns: {list(row.keys())}")

        text = to_text(row[field])
        if not text:
            continue

        tl = token_len(tokenizer, text)

        n += 1
        s += tl
        s2 += tl * tl
        mn = min(mn, tl)
        mx = max(mx, tl)

        reservoir_update(stored, tl, n, max_store)

    if n == 0:
        return {
            "field": field, "n": 0, "mean": None, "std": None,
            "min": None, "p50": None, "p95": None, "p99": None, "max": None
        }

    arr = np.array(stored, dtype=np.int32)
    mean = s / n
    var = max(0.0, (s2 / n) - (mean * mean))
    std = math.sqrt(var)

    # Percentiles from stored sample (exact if dataset <= max_store, approximate otherwise)
    p50, p95, p99 = np.percentile(arr, [50, 95, 99]).tolist()

    return {
        "field": field,
        "n": n,
        "mean": float(mean),
        "std": float(std),
        "min": int(mn),
        "p50": float(p50),
        "p95": float(p95),
        "p99": float(p99),
        "max": int(mx),
        "percentiles_note": (
            "exact" if n <= max_store else f"approx (reservoir sample of {max_store})"
        ),
    }


def load_mteb_dataset(dataset_name, subset, split, streaming=STREAMING):
    path = f"mteb/{dataset_name}"
    if subset is None:
        return load_dataset(path, split=split, streaming=streaming)
    return load_dataset(path, subset, split=split, streaming=streaming)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    for dataset_name, subset, split in tasks:
        cols = COLUMNS_BY_DATASET.get(dataset_name, DEFAULT_COLUMNS)

        ds = load_mteb_dataset(dataset_name, subset, split, streaming=STREAMING)

        # For streaming datasets, you can only iterate once, so we reload per field.
        rows_out = []
        for field in cols:
            ds_field = load_mteb_dataset(dataset_name, subset, split, streaming=STREAMING)
            stats = compute_stats_for_field(
                tqdm(ds_field, desc=f"{dataset_name} | {subset or '-'} | {split} | {field}"),
                field=field,
                tokenizer=tokenizer,
            )
            rows_out.append(stats)

        df = pd.DataFrame(rows_out)
        df.insert(0, "dataset_name", dataset_name)
        df.insert(1, "subset", subset)
        df.insert(2, "split", split)
        df.insert(3, "model_name", MODEL_NAME)

        out_name = f"{safe_name(dataset_name)}__{safe_name(subset or 'no_subset')}__{safe_name(split)}.csv"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        df.to_csv(out_path, index=False)
        print(f"Wrote: {out_path}")

In [8]:
main()

Assin2STS | - | test | sentence1: 2448it [00:00, 4194.44it/s]
Assin2STS | - | test | sentence2: 2448it [00:00, 4645.78it/s]


Wrote: ../data/tokens/bertimbau/Assin2STS__no_subset__test.csv


SICK-BR-STS | - | test | sentence1: 2048it [00:00, 4675.66it/s]
SICK-BR-STS | - | test | sentence2: 2048it [00:00, 4642.15it/s]


Wrote: ../data/tokens/bertimbau/SICK-BR-STS__no_subset__test.csv


stsb_multi_mt | pt | test | sentence1: 1379it [00:00, 2720.90it/s]
stsb_multi_mt | pt | test | sentence2: 1379it [00:00, 3196.33it/s]


...

Wrote: ../data/tokens/bertimbau/WebFAQRetrieval__por-corpus__test.csv


MultiLongDocRetrieval | pt-corpus | test | text: 6569it [00:51, 126.34it/s]


Wrote: ../data/tokens/bertimbau/MultiLongDocRetrieval__pt-corpus__test.csv


WikipediaRetrievalMultilingual | pt-corpus | test | text: 13500it [00:02, 5772.90it/s]

Wrote: ../data/tokens/bertimbau/WikipediaRetrievalMultilingual__pt-corpus__test.csv
